
## File 03 — Pipeline Failure Recovery:



This notebook identifies failed pipeline tasks, classifies errors, records recovery actions, and determines whether a task can safely be retried.

It covers:

- Failed task detection.
- Error classification.
- Retryable versus permanent failures.
- Recovery action tracking.
- Recovery execution history.
- Failure summary reporting.


## Recovery Architecture Flow


```text
+----------------------------------+
| PIPELINE TASK FAILS              |
|                                  |
| Bronze / Silver / Gold           |
+----------------------------------+
                 |
                 v
+----------------------------------+
| READ CENTRALIZED EXECUTION LOGS  |
|                                  |
| Execution ID                     |
| Error Message                    |
| Failure Details                  |
+----------------------------------+
                 |
                 v
+----------------------------------+
| CLASSIFY THE ERROR               |
|                                  |
| Retryable or Permanent           |
+----------------------------------+
                 |
          +------+------+
          |             |
          v             v
+----------------+  +----------------+
| RETRYABLE      |  | PERMANENT      |
| ERROR          |  | ERROR          |
+----------------+  +----------------+
          |             |
          v             v
+----------------+  +----------------+
| RETRY WITH     |  | FIX ROOT       |
| EXPONENTIAL    |  | CAUSE           |
| BACKOFF        |  |                 |
+----------------+  +----------------+
          |             |
          v             v
+----------------+  +----------------+
| VERIFY RETRY   |  | RE-RUN         |
| RESULT         |  | AFFECTED TASK  |
+----------------+  +----------------+
          |             |
          +------+------+
                 |
                 v
+----------------------------------+
| RECORD RECOVERY ACTION & OUTCOME |
|                                  |
| Attempts | Status | Timestamps   |
| Error | Final Outcome            |
+----------------------------------+
```

In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timezone
import uuid


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"
GOLD_SCHEMA = "ecommerce_gold"

LOG_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_execution_logs"
)

FAILURE_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_failure_report"
)

RECOVERY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_recovery_history"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)


# ============================================================
# 3. CREATE FAILURE REPORT SCHEMA
# ============================================================

failure_schema = StructType([

    StructField("failure_id", StringType(), False),

    StructField("run_id", StringType(), True),

    StructField("pipeline_name", StringType(), True),

    StructField("task_name", StringType(), True),

    StructField("dataset_name", StringType(), True),

    StructField("pipeline_layer", StringType(), True),

    StructField("error_type", StringType(), True),

    StructField("error_message", StringType(), True),

    StructField("failure_category", StringType(), True),

    StructField("retry_recommended", BooleanType(), True),

    StructField("failure_timestamp", TimestampType(), True),

    StructField("investigation_status", StringType(), True)

])


# ============================================================
# 4. CREATE RECOVERY HISTORY SCHEMA
# ============================================================

recovery_schema = StructType([

    StructField("recovery_id", StringType(), False),

    StructField("failure_id", StringType(), True),

    StructField("run_id", StringType(), True),

    StructField("task_name", StringType(), True),

    StructField("recovery_action", StringType(), True),

    StructField("recovery_status", StringType(), True),

    StructField("attempt_number", IntegerType(), True),

    StructField("recovery_message", StringType(), True),

    StructField("recovery_timestamp", TimestampType(), True)

])


# ============================================================
# 5. CREATE TABLES IF THEY DO NOT EXIST
# ============================================================

if not spark.catalog.tableExists(FAILURE_TABLE):

    (

        spark.createDataFrame(
            [],
            failure_schema
        )

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(FAILURE_TABLE)

    )


if not spark.catalog.tableExists(RECOVERY_TABLE):

    (

        spark.createDataFrame(
            [],
            recovery_schema
        )

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(RECOVERY_TABLE)

    )


# ============================================================
# 6. VERIFY SOURCE LOG TABLE
# ============================================================

if not spark.catalog.tableExists(LOG_TABLE):

    raise ValueError(
        f"Pipeline log table not found: {LOG_TABLE}. "
        "Run Folder 10 File 01 first."
    )


logs_df = spark.table(LOG_TABLE)


# ============================================================
# 7. IDENTIFY FAILED TASKS
# ============================================================

failed_logs_df = (

    logs_df

    .filter(
        F.upper(F.col("execution_status")) == "FAILED"
    )

)


# ============================================================
# 8. CLASSIFY PIPELINE ERRORS
# ============================================================

def classify_error(error_type, error_message):

    error_type = (
        str(error_type or "").lower()
    )

    error_message = (
        str(error_message or "").lower()
    )

    combined_error = (
        error_type + " " + error_message
    )

    # Potentially temporary failures.
    retryable_patterns = [

        "timeout",
        "temporarily unavailable",
        "connection reset",
        "connection refused",
        "network",
        "service unavailable",
        "rate limit",
        "too many requests",
        "concurrent modification",
        "concurrenttransactionexception",
        "temporarily overloaded"

    ]

    # Data and configuration failures.
    permanent_patterns = [

        "analysisexception",
        "syntax error",
        "parseexception",
        "column not found",
        "cannot resolve",
        "permission denied",
        "access denied",
        "authentication",
        "schema mismatch",
        "file not found",
        "invalid input",
        "valueerror",
        "typeerror",
        "assertionerror"

    ]

    if any(
        pattern in combined_error
        for pattern in retryable_patterns
    ):

        return "RETRYABLE"

    if any(
        pattern in combined_error
        for pattern in permanent_patterns
    ):

        return "PERMANENT"

    return "REQUIRES_INVESTIGATION"


# ============================================================
# 9. BUILD FAILURE REPORT
# ============================================================

failure_rows = []

for row in failed_logs_df.collect():

    error_category = classify_error(

        row["error_type"],

        row["error_message"]

    )

    failure_rows.append({

        "failure_id": str(uuid.uuid4()),

        "run_id": row["run_id"],

        "pipeline_name": row["pipeline_name"],

        "task_name": row["task_name"],

        "dataset_name": row["dataset_name"],

        "pipeline_layer": row["pipeline_layer"],

        "error_type": row["error_type"],

        "error_message": row["error_message"],

        "failure_category": error_category,

        "retry_recommended": (
            error_category == "RETRYABLE"
        ),

        "failure_timestamp": row["logged_at"],

        "investigation_status": "PENDING"

    })


# ============================================================
# 10. SAVE FAILURE REPORT
# ============================================================

if failure_rows:

    failure_df = spark.createDataFrame(

        failure_rows,

        schema=failure_schema

    )

    (

        failure_df

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(FAILURE_TABLE)

    )

    print(
        f"Detected failures: {len(failure_rows)}"
    )

else:

    print("No failed pipeline tasks found.")


# ============================================================
# 11. RECOVERY ACTION DECISION
# ============================================================

def recommend_recovery_action(failure_category):

    if failure_category == "RETRYABLE":

        return "RETRY_AFTER_BACKOFF"

    elif failure_category == "PERMANENT":

        return "FIX_ROOT_CAUSE"

    else:

        return "MANUAL_INVESTIGATION"


# ============================================================
# 12. CREATE RECOVERY EXECUTION FUNCTION
# ============================================================

def record_recovery_action(

    failure_id,

    run_id,

    task_name,

    recovery_action,

    recovery_status,

    attempt_number,

    recovery_message

):

    recovery_record = {

        "recovery_id": str(uuid.uuid4()),

        "failure_id": failure_id,

        "run_id": run_id,

        "task_name": task_name,

        "recovery_action": recovery_action,

        "recovery_status": recovery_status,

        "attempt_number": int(attempt_number),

        "recovery_message": str(
            recovery_message
        )[:4000],

        "recovery_timestamp": (
            datetime.now(timezone.utc).replace(
                tzinfo=None
            )
        )

    }

    recovery_df = spark.createDataFrame(

        [recovery_record],

        schema=recovery_schema

    )

    (

        recovery_df

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(RECOVERY_TABLE)

    )


# ============================================================
# 13. BUILD RECOVERY RECOMMENDATIONS
# ============================================================

if failure_rows:

    recommendations = []

    for failure in failure_rows:

        action = recommend_recovery_action(

            failure["failure_category"]

        )

        recommendations.append({

            "failure_id": failure["failure_id"],

            "run_id": failure["run_id"],

            "task_name": failure["task_name"],

            "failure_category": failure["failure_category"],

            "recommended_action": action

        })

    recommendations_df = spark.createDataFrame(
        recommendations
    )

    display(recommendations_df)

else:

    print(
        "No recovery recommendations required."
    )


# ============================================================
# 14. DEMONSTRATE SAFE RETRY WRAPPER
# ============================================================

import time


def execute_with_retry(

    task_function,

    max_attempts=3,

    initial_delay_seconds=5

):

    last_error = None

    for attempt in range(1, max_attempts + 1):

        try:

            result = task_function()

            return {

                "status": "SUCCESS",

                "attempt_number": attempt,

                "result": result

            }

        except Exception as error:

            last_error = error

            print(
                f"Attempt {attempt} failed: {error}"
            )

            if attempt < max_attempts:

                delay = (
                    initial_delay_seconds * (2 ** (attempt - 1))
                )

                print(
                    f"Retrying after {delay} seconds..."
                )

                time.sleep(delay)

    return {

        "status": "FAILED",

        "attempt_number": max_attempts,

        "error_message": str(last_error)

    }


# ============================================================
# 15. DEMONSTRATION TASK
# ============================================================

# This is a harmless demonstration.
# It does not modify Bronze, Silver or Gold data.

def demonstration_task():

    return "Task completed successfully."


retry_result = execute_with_retry(

    task_function=demonstration_task,

    max_attempts=3,

    initial_delay_seconds=2

)

print(retry_result)


# ============================================================
# 16. FAILURE SUMMARY
# ============================================================

if spark.catalog.tableExists(FAILURE_TABLE):

    failure_summary = (

        spark.table(FAILURE_TABLE)

        .groupBy(
            "failure_category",
            "investigation_status"
        )

        .agg(

            F.count("*").alias("failure_count")

        )

        .orderBy(
            "failure_category"
        )

    )

    display(failure_summary)


# ============================================================
# 17. RECOVERY HISTORY
# ============================================================

display(

    spark.table(RECOVERY_TABLE)

    .orderBy(
        F.col("recovery_timestamp").desc()
    )

)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)

print("PIPELINE FAILURE RECOVERY SUMMARY")

print("=" * 65)

print(
    f"New failures detected: {len(failure_rows)}"
)

print(
    f"Failure report table: {FAILURE_TABLE}"
)

print(
    f"Recovery history table: {RECOVERY_TABLE}"
)

print("=" * 65)

print("FAILURE DETECTION COMPLETED.")



## View failures

In [0]:
%sql

SELECT
    failure_id,
    run_id,
    task_name,
    pipeline_layer,
    failure_category,
    retry_recommended,
    error_message,
    failure_timestamp
FROM workspace.ecommerce_gold.pipeline_failure_report
ORDER BY failure_timestamp DESC;


## Count failures by category

In [0]:
%sql

SELECT
    failure_category,
    COUNT(*) AS total_failures
FROM workspace.ecommerce_gold.pipeline_failure_report
GROUP BY failure_category;


### View recovery history

In [0]:
%sql

SELECT
    failure_id,
    task_name,
    recovery_action,
    recovery_status,
    attempt_number,
    recovery_message,
    recovery_timestamp
FROM workspace.ecommerce_gold.pipeline_recovery_history
ORDER BY recovery_timestamp DESC;